# Lectra AI — GPU tunnel worker

Runs the project's real audio pipeline (`LectraAIPipeline`, unmodified) on Kaggle's free GPU, and exposes it to your local backend over a public tunnel URL.

**Before running:** Notebook Settings (right sidebar) → Accelerator → **GPU T4 x2** (or P100) → Internet → **On**.

**Also before running:** Add-ons → Secrets → attach two secrets to this notebook:
- `GPU_TUNNEL_TOKEN` — any long random string you make up. This is the shared password your local backend uses to talk to this worker; anyone with the tunnel URL *and* this token can submit jobs.
- `HF_TOKEN` — your HuggingFace token (needed for speaker diarization), same one from your local `.env`.

Run every cell top to bottom, then copy the printed tunnel URL (and the token you chose) into your local `.env` as `GPU_TUNNEL_URL` / `GPU_TUNNEL_TOKEN`, and restart your local backend. See `gpu_tunnel/README.md` for the full walkthrough.

Session limits (Kaggle free tier, subject to change): ~9-12h per session, ~30 GPU-hours/week. The tunnel URL changes every time you restart this notebook — just paste the new one in.

In [ ]:
# --- 1. Confirm we actually have a GPU before doing anything else ---
import torch

assert torch.cuda.is_available(), (
    "No GPU detected! Notebook Settings (right sidebar) -> Accelerator -> "
    "GPU T4 x2 (or P100). Also check Settings -> Internet -> On."
)
print(f"GPU OK: {torch.cuda.get_device_name(0)}")

In [ ]:
# --- 2. Get the actual pipeline code (public repo, no auth needed) ---
# cd to a stable parent FIRST, then rm -rf + clone - never delete a
# directory that is an ancestor of the shell's current cwd (that leaves
# the shell broken on re-runs within the same session).
%cd /kaggle/working
!rm -rf repo && git clone --depth 1 https://github.com/Hanzala-12/lectra-ai.git repo
%cd /kaggle/working/repo

In [ ]:
# --- 3. Install ONLY what is missing from Kaggle's own image ---
# kaggle_requirements.txt deliberately excludes torch/torchaudio/numpy/scipy
# so pip's resolver leaves Kaggle's preinstalled CUDA build alone.
#
# Three things this needs that are easy to get wrong (found the hard way):
# 1. Always install via {sys.executable} -m pip, never bare "pip install" -
#    on Kaggle the plain pip on PATH can resolve to a DIFFERENT Python than
#    the one this notebook's kernel actually runs (confirmed: bare
#    "pip install" reported success while the kernel's own interpreter
#    still couldn't import any of it).
# 2. deepfilternet depends on DeepFilterLib, a Rust extension with no
#    prebuilt wheel for Kaggle's Python version - it needs an actual Rust
#    toolchain to build from source, which Kaggle doesn't ship by default.
# 3. Even though kaggle_requirements.txt never names numpy, pip's own
#    resolver can still silently downgrade it as a side effect of some
#    *other* package's transitive version constraint (confirmed live:
#    Kaggle's preinstalled numpy 2.0.2 got silently swapped for 1.26.4
#    partway through this same install, with no error or warning) - which
#    then breaks scipy (`ModuleNotFoundError: No module named
#    'numpy.strings'`/`'numpy.char'`/`'numpy.rec'`, since those only exist
#    on numpy>=2.0). Capture numpy's version before this install and force
#    it back afterward - self-adapting rather than a hardcoded version, so
#    this keeps working whatever numpy Kaggle's image ships next.
import sys, os, subprocess

rust_install = subprocess.run(
    ["bash", "-c",
     "curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs "
     "| sh -s -- -y --default-toolchain stable"],
    capture_output=True, text=True,
)
assert rust_install.returncode == 0, rust_install.stderr[-2000:]
os.environ["PATH"] = os.path.expanduser("~/.cargo/bin") + ":" + os.environ["PATH"]
print("Rust toolchain installed.")

import numpy
_numpy_version_before = numpy.__version__

install = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-r", "gpu_tunnel/kaggle_requirements.txt"],
    capture_output=True, text=True, env=os.environ,
)
if install.returncode != 0:
    print("First install attempt failed, retrying once...")
    install = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-r", "gpu_tunnel/kaggle_requirements.txt"],
        capture_output=True, text=True, env=os.environ,
    )
print(install.stdout[-3000:])
assert install.returncode == 0, install.stderr[-2000:]

restore = subprocess.run(
    [sys.executable, "-m", "pip", "install", "--force-reinstall", "--no-deps",
     f"numpy=={_numpy_version_before}"],
    capture_output=True, text=True,
)
assert restore.returncode == 0, restore.stderr[-2000:]
print(f"numpy restored to its pre-install version ({_numpy_version_before}).")

import torch
assert torch.cuda.is_available(), (
    "GPU was available before installing requirements but NOT after - "
    "something in kaggle_requirements.txt pulled in a CPU-only torch build. "
    "Do not add torch/torchaudio/numpy/scipy to that file."
)
print(f"Still GPU OK after installs: {torch.cuda.get_device_name(0)}")

In [ ]:
# --- 4. Load secrets (see the Add-ons -> Secrets setup note at the top) ---
import os
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()

try:
    os.environ["GPU_TUNNEL_TOKEN"] = secrets.get_secret("GPU_TUNNEL_TOKEN")
except Exception:
    raise RuntimeError(
        "Couldn't read the GPU_TUNNEL_TOKEN secret. Add-ons -> Secrets -> "
        "add one named exactly GPU_TUNNEL_TOKEN (any random string you make "
        "up) -> make sure it's attached/enabled for THIS notebook."
    )

try:
    os.environ["HF_TOKEN"] = secrets.get_secret("HF_TOKEN")
except Exception:
    print(
        "No HF_TOKEN secret found - diarization will fall back to VAD "
        "(one undifferentiated 'speaker'), same as running locally without "
        "it. Add one via Add-ons -> Secrets if you want real diarization."
    )

print("Secrets loaded.")

In [ ]:
# --- 5. Start the worker API in the background ---
import threading
import time
import asyncio
import traceback
import uvicorn
import uvicorn.config

from gpu_tunnel.worker_app import app

# Kaggle's preinstalled uvicorn (confirmed: 0.46.0, from
# /usr/local/lib/python3.12/dist-packages) ships an internally-
# inconsistent build: Config.get_loop_factory() (the method uvicorn.run()
# actually calls to pick an event loop, replacing the older
# setup_event_loop() as of upstream uvicorn 0.36.0) expects each
# uvicorn/loops/<name>.py submodule to expose a `<name>_loop_factory`
# callable - but the loops/ submodules actually installed on this image
# are stale pre-0.36.0 files that only define the old `asyncio_setup`/etc
# names. Patching it to just return a plain stdlib event-loop factory
# sidesteps the broken lookup entirely.
uvicorn.config.Config.get_loop_factory = lambda self: asyncio.new_event_loop

_server_error = None


def _run_server():
    global _server_error
    try:
        uvicorn.run(app, host="0.0.0.0", port=8800, log_level="info")
    except Exception:
        _server_error = traceback.format_exc()


threading.Thread(target=_run_server, daemon=True).start()
time.sleep(8)

# uvicorn.run() blocks forever once it's actually serving, so
# _server_error staying None after the sleep above is the normal
# success case. But a bare background thread's uncaught exception is
# otherwise completely silent - this cell would show as "completed" with
# no visible error even if the server never actually came up (confirmed
# live: exactly this happened once, with no clue why until dug into it
# from the Console). So explicitly surface any captured exception, AND
# confirm the port is actually answering, before declaring success -
# don't just trust that the cell finished without a top-level error.
if _server_error:
    print(_server_error)
    raise RuntimeError("Worker API thread raised - see traceback above.")

import requests

try:
    requests.get("http://localhost:8800/docs", timeout=5)
    print("Worker API confirmed running on port 8800.")
except Exception as e:
    raise RuntimeError(
        f"Worker API port 8800 isn't responding ({e}). It may still be "
        "starting up (loading models) - wait a few seconds and re-run "
        "this cell, or check Kaggle's own session logs for what actually "
        "happened inside the background thread."
    )

In [ ]:
# --- 6. Open the tunnel (cloudflared quick tunnel - no account needed) ---
import re
import subprocess

!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared

proc = subprocess.Popen(
    ["./cloudflared", "tunnel", "--url", "http://localhost:8800"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

print("Waiting for cloudflared to establish the tunnel...")
url = None
for line in proc.stdout:
    if "trycloudflare.com" in line:
        m = re.search(r"https://[a-zA-Z0-9\-]+\.trycloudflare\.com", line)
        if m:
            url = m.group(0)
            break

print("\n" + "=" * 70)
print(f"TUNNEL URL:  {url}")
print("=" * 70)
print("Paste into your local .env:")
print(f"  GPU_TUNNEL_URL={url}")
print("  GPU_TUNNEL_TOKEN=<the same value you put in the GPU_TUNNEL_TOKEN secret above>")
print("Then restart your local backend (python backend.py).")

In [ ]:
# --- 7. Keep this cell running to keep the worker + tunnel alive ---
# Stop this cell (or close the notebook) to shut the worker down - your
# local backend will automatically fall back to local CPU processing.
import time

print("Worker is live. Keep this cell running - stopping it ends the tunnel.")
while True:
    time.sleep(300)
    print(f"[{time.strftime('%H:%M:%S')}] still running...")